In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
import urllib
from bs4 import BeautifulSoup
from sklearn.manifold import TSNE

In [3]:
def grab_html(url: str):
    """
    Leverages `urllib` to return HTTPResponse object.
    """
    # Set a User-Agent to mimic a browser, otherwise 403 forbidden
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
    }
    # Create a request with the headers
    request = urllib.request.Request(url, headers=headers)
    # open url
    html = urllib.request.urlopen(request)
    return(html)
    

In [4]:
def parse_html(html_object):
    """
    Leverages `BeautifulSoup` to return a soup object
    """
    soup = BeautifulSoup(html_object, 'lxml')
    return(soup)

In [5]:
def extract_town_data(soup):
    """
    Extracts all rows from Wikipedia 'wikitable' tables,
    capturing Town, County/Unitary Authority, and Notes.
    Returns a list of dicts for structured data.
    """
    town_data = []
    # grab each table on the wikipedia page
    for table in soup.select("table.wikitable"):
        # select each row in that table
        for row in table.select("tr")[1:]:  # skip header row
            # grab each set of cells
            cells = row.find_all("td")
            if len(cells) >= 3:
                town = cells[0].get_text(strip=True)
                county = cells[1].get_text(strip=True)
                notes = cells[2].get_text(strip=True)
                town_data.append({
                    "town": town,
                    "county": county,
                    "notes": notes
                })
    return town_data

In [6]:
# british_cities = "https://simple.wikipedia.org/wiki/List_of_cities_in_the_United_Kingdom"
british_towns = "https://en.wikipedia.org/wiki/List_of_towns_in_England"

In [7]:
british_towns_html = grab_html(british_towns)

In [8]:
# parse the html
british_towns_soup = parse_html(british_towns_html)

In [9]:
# british_cities_text = british_cities_soup.get_text()
# british_towns_text = british_towns_soup.get_text()

In [10]:
# Extract the data from the towns
towns = extract_town_data(british_towns_soup)
print(f"Extracted {len(towns)} towns with metadata.")
print(towns[:5])  # show sample


Extracted 980 towns with metadata.
[{'town': 'Abingdon-on-Thames', 'county': 'Oxfordshire', 'notes': 'town council1'}, {'town': 'Accrington', 'county': 'Lancashire', 'notes': 'borough (1878–1974)'}, {'town': 'Acle', 'county': 'Norfolk', 'notes': 'market charter'}, {'town': 'Acton', 'county': 'Greater London', 'notes': 'borough (1921–1965)'}, {'town': 'Adlington', 'county': 'Lancashire', 'notes': 'town council1'}]


In [11]:
british_towns_df = pd.DataFrame(towns)
british_towns_df.head(44)

,town,county,notes
0,Abingdon-on-Thames,Oxfordshire,town council1
1,Accrington,Lancashire,borough (1878–1974)
2,Acle,Norfolk,market charter
3,Acton,Greater London,borough (1921–1965)
4,Adlington,Lancashire,town council1
5,Alcester,Warwickshire,town council
6,Aldeburgh,Suffolk,town council1
7,Aldershot,Hampshire,borough (1922–1974)
8,Alford,Lincolnshire,town council1
9,Alfreton,Derbyshire,town council


In [12]:
from sklearn.feature_extraction.text import CountVectorizer


towns = list(british_towns_df["town"])

# Create ngrams of len 3 (trigrams)
vectorizer = CountVectorizer(analyzer='char', ngram_range=(3,3), lowercase=True)
town_ngrams = vectorizer.fit_transform(towns)

# See what trigrams were found
print(vectorizer.get_feature_names_out())

[' & ' ' ab' ' al' ... 'zes' 'zio' 'zou']


In [13]:
x = pd.DataFrame(
    town_ngrams.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=towns
)

In [14]:
# Convert sparse matrix to dense array
town_dense = town_ngrams.toarray()

# Run t-SNE to reduce high-dimensional trigram data to 2D
tsne = TSNE(n_components=2, perplexity=20, random_state=42, learning_rate=200)
coords = tsne.fit_transform(town_dense)

In [15]:
import plotly
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode, iplot

In [16]:
# # Store results in a DataFrame for convenience
# df_tsne = pd.DataFrame(coords, columns=['x', 'y'])
# df_tsne['town'] = towns

# # Plot
# plt.figure(figsize=(10, 8))
# plt.scatter(df_tsne['x'], df_tsne['y'], alpha=0.7)

# # Add labels for each town
# for i, txt in enumerate(df_tsne['town']):
#     plt.text(df_tsne['x'][i] + 0.5, df_tsne['y'][i], txt, fontsize=9)

# plt.title("t-SNE Linguistic Map of British Town Names")
# plt.xlabel("t-SNE Dimension 1")
# plt.ylabel("t-SNE Dimension 2")
# plt.show()

In [23]:
import pacmap.pacmap as P

# reducer = P.LocalMAP()
reducer = P.PaCMAP()


coords = reducer.fit_transform(town_dense)

In [24]:
import pandas as pd
import plotly.graph_objects as go

# assume coords (Nx2 array-like) and towns (list of labels) already exist
df_tsne = pd.DataFrame(coords, columns=['x', 'y'])
df_tsne['town'] = towns

# scatter points
scatter = go.Scatter(
    x=df_tsne['x'],
    y=df_tsne['y'],
    mode='markers',
    text=df_tsne['town'],
    textposition='middle right',   # places label to the right of marker
    marker=dict(size=8, opacity=0.1),
    hovertext=df_tsne['town'],
    hoverinfo='text'
)

fig = go.Figure(data=[scatter])

fig.update_layout(
    title="t-SNE Linguistic Map of British Town Names",
    xaxis_title="t-SNE Dimension 1",
    yaxis_title="t-SNE Dimension 2",
    width=800,
    height=640
)

fig.show()